# KL divergence table

$D_{KL}$ (nats) between the per-tutor scaffolding-appropriate (S) and
rigor-appropriate (R) action distributions, computed in both directions,
for human tutors and each LM under plain and evaluation-aware prompts.
Laplace add-1 smoothing on pseudo-counts so KL stays finite on small LM
samples.

Consumes the classified facet CSVs produced by `tutorsim taxonomy classify`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from tutorsim import taxonomy as tx

REPO = Path.cwd()
while not (REPO / ".git").exists() and REPO != REPO.parent:
    REPO = REPO.parent

# Update these to point at the classified.csv files produced by
#   tutorsim taxonomy classify --output <dir>
HUMAN_CLASSIFIED = REPO / "data" / "taxonomy" / "human" / "classified.csv"
LM_CLASSIFIED    = REPO / "data" / "taxonomy" / "lm"    / "classified.csv"
OUT_TEX          = REPO / "analysis" / "working-paper-20260630" / "kl_divergence_table.tex"


In [ ]:
# Display order in the table matches the paper figures.
MODEL_DISPLAY = {
    "claude-opus-4-8":             "Claude Opus 4.8",
    "claude-sonnet-4-6":           "Claude Sonnet 4.6",
    "deepseek-ai/DeepSeek-V4-Pro": "DeepSeek V4 Pro",
    "gemini-2.5-pro":              "Gemini 2.5 Pro",
    "gemini-3.5-flash":            "Gemini 3.5 Flash",
    "gpt-5.5-2026-04-23":          "GPT 5.5",
    "gpt-5.4-mini-2026-03-17":     "GPT 5.4 mini",
}
PROMPTS = [("plain", "Plain prompt"),
           ("scaffolding_rigor", "Evaluation-aware prompt")]
LETTERS = tx.CATEGORY_LETTERS  # A..M


In [ ]:
human = tx.read_classified_csv(HUMAN_CLASSIFIED)
lm    = tx.read_classified_csv(LM_CLASSIFIED)

human_df = tx.facets_to_dataframe(human)
lm_df    = tx.facets_to_dataframe(lm)

# Per-(situation, letter) macro % for human; per-(model, prompt, situation,
# letter) macro % for LMs. Both come back long-format from the helper.
h_sit = tx.macro_distribution(human_df, group_keys=("situation_label",))
lm_sit = tx.macro_distribution(
    lm_df, group_keys=("model", "prompt", "situation_label")
)
print(f"human rows: {len(h_sit)}; lm rows: {len(lm_sit)}")


In [ ]:
def smoothed_probs(pct_row, n_moments, alpha=1.0):
    counts = (pct_row.to_numpy() / 100.0) * n_moments
    smoothed = counts + alpha
    return smoothed / smoothed.sum()

def kl_nats(p, q):
    """KL(p || q) in nats (natural log)."""
    return float(np.sum(p * (np.log(p) - np.log(q))))

def vec(df, letter_col="letter", value_col="mean_pct"):
    return (df.set_index(letter_col)[value_col]
              .reindex(LETTERS, fill_value=0.0))


In [ ]:
def human_kl():
    """KL(S||R) and KL(R||S) for the human pool. Prompt-independent."""
    rows_s = h_sit[h_sit["situation_label"] == "scaffolding"]
    rows_r = h_sit[h_sit["situation_label"] == "rigor"]
    n_s = int(rows_s["n_moments"].iloc[0])
    n_r = int(rows_r["n_moments"].iloc[0])
    p = smoothed_probs(vec(rows_s), n_s)
    q = smoothed_probs(vec(rows_r), n_r)
    return kl_nats(p, q), kl_nats(q, p)

def lm_kl(model_id, prompt):
    sub = lm_sit[(lm_sit["model"] == model_id) & (lm_sit["prompt"] == prompt)]
    rows_s = sub[sub["situation_label"] == "scaffolding"]
    rows_r = sub[sub["situation_label"] == "rigor"]
    if rows_s.empty or rows_r.empty:
        return float("nan"), float("nan")
    n_s = int(rows_s["n_moments"].iloc[0])
    n_r = int(rows_r["n_moments"].iloc[0])
    p = smoothed_probs(vec(rows_s), n_s)
    q = smoothed_probs(vec(rows_r), n_r)
    return kl_nats(p, q), kl_nats(q, p)

h_sr, h_rs = human_kl()
rows = [("Human tutors", h_sr, h_rs, h_sr, h_rs)]
for model_id, label in MODEL_DISPLAY.items():
    p_sr, p_rs = lm_kl(model_id, "plain")
    e_sr, e_rs = lm_kl(model_id, "scaffolding_rigor")
    rows.append((label, p_sr, p_rs, e_sr, e_rs))

kl_table = pd.DataFrame(
    rows,
    columns=["Tutor",
             "plain_S||R", "plain_R||S",
             "eval_S||R",  "eval_R||S"],
)
kl_table.round(3)


## Emit LaTeX

Token-replacement template. Cells use `\gradientwo{...}` (defined in the
paper preamble) so cell colour scales with magnitude. Human row sits at
the top with a `\midrule` separating it from the LM block; LM rows
follow the paper's tutor order.

In [ ]:
def f3(x):
    return f"{x:.3f}" if x == x else "--"  # NaN-safe

HEADER = (
    r"\begin{table}[H]" "\n"
    r"    \centering" "\n"
    r"    \begin{tabular}{lcc cc}" "\n"
    r"        \toprule" "\n"
    r"        & \multicolumn{2}{c}{\textbf{Plain prompt}} & \multicolumn{2}{c}{\textbf{Evaluation-aware prompt}} \\" "\n"
    r"        \cmidrule(lr){2-3} \cmidrule(lr){4-5}" "\n"
    r"        \textbf{Tutor} & $D_\text{KL}(\text{S} \| \text{R})$ & $D_\text{KL}(\text{R} \| \text{S})$ & $D_\text{KL}(\text{S} \| \text{R})$ & $D_\text{KL}(\text{R} \| \text{S})$ \\" "\n"
    r"        \midrule" "\n"
)
FOOTER = (
    r"        \bottomrule" "\n"
    r"    \end{tabular}" "\n"
    r"    \caption{KL divergence between scaffolding-appropriate (S) and rigor-appropriate moment (R) distributions, computed in both directions, for human tutors and each LM tutor under plain and evaluation-aware prompts.}" "\n"
    r"    \label{tab:kl_divergence}" "\n"
    r"\end{table}" "\n"
)

def fmt_row(name, p_sr, p_rs, e_sr, e_rs, *, pad=17):
    name_p = name.ljust(pad)
    return (
        f"        {name_p} & "
        f"\\gradientwo{{{f3(p_sr)}}} & \\gradientwo{{{f3(p_rs)}}} & "
        f"\\gradientwo{{{f3(e_sr)}}} & \\gradientwo{{{f3(e_rs)}}} \\\\\n"
    )

body = fmt_row(*kl_table.iloc[0])
body += r"        \midrule" + "\n"
for _, r in kl_table.iloc[1:].iterrows():
    body += fmt_row(r["Tutor"], r["plain_S||R"], r["plain_R||S"],
                    r["eval_S||R"], r["eval_R||S"])

tex = HEADER + body + FOOTER
OUT_TEX.write_text(tex)
print(f"wrote {OUT_TEX.relative_to(REPO)}")
print()
print(tex)


In [ ]:
"""Inline preview: render the .tex if a LaTeX toolchain (and the
\gradientwo macro) is available, else show an HTML mirror of the same
values so the preview can never drift from the .tex."""
import os, shutil, subprocess, tempfile
from pathlib import Path as _P
from IPython.display import Image, HTML, display

def _find(name, *extra):
    return shutil.which(name) or next((c for c in extra if os.path.exists(c)), None)

def render_latex_png(body):
    pdflatex = _find("pdflatex", "/Library/TeX/texbin/pdflatex")
    if not pdflatex:
        return None
    with tempfile.TemporaryDirectory() as tmp:
        tex = _P(tmp) / "t.tex"
        # Provide a minimal \gradientwo so the standalone test compiles.
        tex.write_text(
            "\\documentclass[preview,border=8pt]{standalone}\n"
            "\\usepackage{booktabs,float}\n"
            "\\newcommand{\\gradientwo}[1]{#1}\n"
            "\\begin{document}\n" + body + "\\end{document}\n"
        )
        try:
            subprocess.run([pdflatex, "-interaction=nonstopmode", "t.tex"],
                           cwd=tmp, check=True, capture_output=True)
        except Exception:
            return None
        pdf = _P(tmp) / "t.pdf"
        for conv in [["pdftoppm", "-r", "180", str(pdf), str(_P(tmp)/"t"), "-png"],
                     ["convert", "-density", "180", str(pdf), str(_P(tmp)/"t.png")]]:
            if shutil.which(conv[0]):
                subprocess.run(conv, check=False, capture_output=True)
                cand = list(_P(tmp).glob("t*.png"))
                if cand: return cand[0].read_bytes()
    return None

png = render_latex_png(tex)
if png:
    display(Image(data=png))
else:
    display(HTML(kl_table.round(3).to_html(index=False)))
